# Can local models file claims hc accepts?

Runs `hc probe-model` against Ollama on a GPU (Colab, Kaggle, or a local RTX 3070 Ti). For each model it compares the claim schema enforced at decode time (Ollama `format`) with prompt-only output, and reports how often the first answer is valid and how often it is also correct.

Models below are 7–8B and fit a 16 GB T4 or an 8 GB 3070 Ti at their default quantization.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Ollama's official installer, then the server in the background.
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

In [ ]:
MODELS = ["qwen3:8b", "llama3.1:8b", "qwen2.5-coder:7b"]
for m in MODELS:
    !ollama pull {m}

In [ ]:
# Rust toolchain and the hc source. Set REPO_URL to your clone, or upload
# handoff-compiler.zip to the notebook's working directory instead.
REPO_URL = ""
!curl https://sh.rustup.rs -sSf | sh -s -- -y --profile minimal
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
if REPO_URL:
    !git clone --depth 1 {REPO_URL} handoff-compiler
elif os.path.exists("handoff-compiler.zip"):
    !unzip -q -o handoff-compiler.zip
!cd handoff-compiler && cargo build --release -p hc 2>&1 | tail -2

In [ ]:
args = " ".join(f"--model {m}" for m in MODELS)
!./handoff-compiler/target/release/hc probe-model {args} --repeats 3 > probe.json

In [ ]:
import json, pandas as pd
rows = json.load(open("probe.json"))
df = pd.DataFrame([{ "model": r["model"], "mode": "schema" if r["constrained"] else "prompt-only",
                     "valid %": round(100 * r["valid"] / r["trials"]), "correct %": round(100 * r["semantic"] / r["trials"]),
                     "median ms": r["median_ms"], "trials": r["trials"] } for r in rows])
df.sort_values(["model", "mode"])

In [ ]:
# The scenarios each model got wrong, with the first diagnostic.
for r in rows:
    if r["failures"]:
        print(r["model"], "schema" if r["constrained"] else "prompt-only")
        for scenario, why in r["failures"][:6]:
            print("   ", scenario, "->", why)